# Bronze → Silver · V-Credit

Mantém a Bronze recebida e entrega as oito bases tratadas, mais `tb_log_qualidade`.
A Silver preserva o grão de cada base; custos, pesquisas e jornadas não são somados
dentro dos chamados. Esses relacionamentos e indicadores pertencem à Gold.

Execute **Run all**, depois do sucesso completo da Bronze. Não usa funções de janela,
UDF, RDD ou pandas. Os metadados de origem são mantidos nas oito tabelas.

In [ ]:
from pyspark.sql import functions as F

catalogo = "vcredit"
bronze = f"{catalogo}.bronze"
silver = f"{catalogo}.silver"
metadados = ["ingestion_datetime", "arquivo_origem"]

spark.conf.set("spark.sql.session.timeZone", "America/Sao_Paulo")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver}")
log_qualidade = {}


def vazio(coluna):
    return coluna.isNull() | (F.trim(coluna) == "")


def chave_texto(coluna):
    # Normalização para associação, sem mudar o significado do texto original.
    acentos = "áàâãäéèêëíìîïóòôõöúùûüç"
    letras = "aaaaaeeeeiiiiooooouuuuc"
    return F.trim(F.regexp_replace(F.translate(F.lower(coluna), acentos, letras), r"\s+", " "))


def de_para(dicionario):
    return F.create_map(*[F.lit(valor) for par in dicionario.items() for valor in par])


def data_segura(coluna):
    return F.coalesce(
        F.to_date(F.try_to_timestamp(F.trim(coluna), F.lit("dd/MM/yyyy"))),
        F.to_date(F.try_to_timestamp(F.trim(coluna), F.lit("yyyy-MM-dd"))),
    )


def registrar(base, regra, afetadas, total, tipo="correcao", etapa="tratamento"):
    # A mesma regra atualiza seu registro se uma célula for executada novamente.
    log_qualidade[(base, regra)] = (int(afetadas), int(total), tipo, etapa)


def registrar_condicoes(df, base, regras, etapa="tratamento"):
    # Conta várias regras em uma consulta. Só o resumo vai para o Python.
    resumo = df.agg(F.count("*").alias("total"), *[
        F.sum(F.when(F.coalesce(condicao, F.lit(False)), 1).otherwise(0)).alias(f"regra_{i}")
        for i, (_, _, condicao) in enumerate(regras)
    ]).first()
    for i, (regra, tipo, _) in enumerate(regras):
        registrar(base, regra, resumo[f"regra_{i}"] or 0, resumo["total"], tipo, etapa)


def salvar_log():
    linhas = [(base, regra, *valores) for (base, regra), valores in sorted(log_qualidade.items())]
    df_log = spark.createDataFrame(
        linhas,
        "base string, regra string, linhas_afetadas long, total_linhas long, tipo string, etapa string",
    ).withColumn(
        "percentual", F.when(F.col("total_linhas") > 0,
                            F.round(100.0 * F.col("linhas_afetadas") / F.col("total_linhas"), 2))
    ).withColumn("data_execucao", F.current_timestamp())
    df_log.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver}.tb_log_qualidade")


def exigir(df, base, condicao_invalida, regra, total_base=None):
    total = df.count() if total_base is None else total_base
    afetadas = df.filter(F.coalesce(condicao_invalida, F.lit(False))).count()
    registrar(base, regra, afetadas, total, "bloqueio", "validacao")
    if afetadas:
        # Salva o diagnóstico antes de interromper. Não escolhe registros ao acaso.
        salvar_log()
        raise ValueError(f"{base}: {regra}; {afetadas} linhas. Consulte tb_log_qualidade.")


def validar_unicidade(df, base, chaves):
    falta = None
    for chave in chaves:
        condicao = F.col(chave).isNull()
        falta = condicao if falta is None else falta | condicao
    exigir(df, base, falta, "Chave obrigatoria ausente ou invalida")
    conflitos = df.groupBy(*chaves).count().filter(F.col("count") > 1)
    afetadas = conflitos.agg(F.sum("count")).first()[0] or 0
    registrar(base, "Linhas em chaves repetidas com conflito", afetadas, df.count(), "bloqueio", "validacao")
    if afetadas:
        salvar_log()
        raise ValueError(f"{base}: chave repetida com conteudos diferentes; falta uma regra de escolha.")


def carregar(base):
    if not spark.catalog.tableExists(f"{bronze}.{base}"):
        raise ValueError(f"Falta {bronze}.{base}. Execute a Bronze completa.")
    df = spark.table(f"{bronze}.{base}")
    if not all(coluna in df.columns for coluna in metadados):
        raise ValueError(f"{base}: faltam metadados da Bronze recebida.")
    # Deduplica somente linhas de negócio idênticas. A Bronze é um snapshot completo,
    # com metadados iguais dentro de cada carga; metadados não definem a duplicidade.
    negocio = [c for c in df.columns if c not in metadados]
    total = df.count()
    resultado = df.dropDuplicates(negocio)
    registrar(base, "Linhas de negocio identicas removidas", total - resultado.count(), total, "descarte", "origem")
    return resultado


def gravar_silver(df, tabela):
    total = df.count()
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver}.{tabela}")
    if spark.table(f"{silver}.{tabela}").count() != total:
        raise ValueError(f"{tabela}: quantidade divergente depois da gravacao.")
    print(f"OK: {silver}.{tabela}: {total} linhas.")

# Confere as oito entradas antes de iniciar a substituição de tabelas Silver.
# A Bronze não é modificada por este notebook.
colunas_bronze = {
    "tb_canais": "id_canal nome_canal tipo_canal status_canal",
    "tb_motivos": "id_motivo nome_motivo categoria criticidade",
    "tb_atendentes": "id_atendente nome_atendente nivel_atendimento data_admissao data_desligamento status_atendente",
    "tb_clientes": "id_cliente nome email regiao idade data_cadastro dia_vencimento_fatura",
    "tb_chamados_hora": "id_chamado id_cliente hora_abertura_chamado hora_inicio_atendimento hora_finalizacao_atendimento",
    "tb_chamados": "id_chamado id_cliente motivo canal resolvido hora_abertura_chamado hora_inicio_atendimento hora_finalizacao_atendimento tempo_espera tempo_atendimento id_atendente",
    "tb_custos": "id_custo id_chamado custo",
    "tb_pesquisa_satisfacao": "id_pesquisa id_chamado nota_atendimento",
}
for tabela, colunas in colunas_bronze.items():
    if not spark.catalog.tableExists(f"{bronze}.{tabela}"):
        raise ValueError(f"Execute a Bronze completa: falta {tabela}.")
    campos = {campo.name: campo.dataType.simpleString() for campo in spark.table(f"{bronze}.{tabela}").schema.fields}
    if any(c not in campos for c in colunas.split() + metadados):
        raise ValueError(f"{tabela}: faltam colunas do contrato ou metadados.")
    if any(campos[c] != "string" for c in colunas.split()):
        raise ValueError(f"{tabela}: colunas da Bronze devem continuar como texto.")


## 1. Canais

Padronizamos nome, tipo, nível e status. IDs, nomes ou domínios desconhecidos
bloqueiam a carga; não removemos uma versão diferente só porque repete um ID.

In [ ]:
nomes_canais = {
    "ura": "URA", "chatbot": "Chatbot",
    "atendimentoinicial": "Atendimento Inicial", "atendinicial": "Atendimento Inicial",
    "atendimentoespecializado": "Atendimento Especializado", "atendespecializado": "Atendimento Especializado",
}
tipos_canais = {"autosservico": "Autosserviço", "humano": "Humano"}
niveis_canais = {"URA": "Autosserviço", "Chatbot": "Autosserviço", "Atendimento Inicial": "N1", "Atendimento Especializado": "N2"}

df = carregar("tb_canais")
registrar_condicoes(df, "tb_canais", [
    ("Nome do canal padronizado", "correcao", ~F.col("nome_canal").eqNullSafe(de_para(nomes_canais)[F.regexp_replace(chave_texto(F.col("nome_canal")), "[^a-z0-9]", "")])),
    ("Tipo do canal padronizado", "correcao", ~F.col("tipo_canal").eqNullSafe(de_para(tipos_canais)[chave_texto(F.col("tipo_canal"))])),
    ("Status do canal padronizado", "correcao", ~F.col("status_canal").eqNullSafe(chave_texto(F.col("status_canal")))),
])
df_canais = df.select(
    F.expr("try_cast(trim(id_canal) AS INT)").alias("id_canal"),
    de_para(nomes_canais)[F.regexp_replace(chave_texto(F.col("nome_canal")), "[^a-z0-9]", "")].alias("nome_canal"),
    de_para(tipos_canais)[chave_texto(F.col("tipo_canal"))].alias("tipo_canal"),
    chave_texto(F.col("status_canal")).alias("status_canal"), *metadados,
).withColumn("nivel_canal", de_para(niveis_canais)[F.col("nome_canal")])
validar_unicidade(df_canais, "tb_canais", ["id_canal"])
exigir(df_canais, "tb_canais", (F.col("id_canal") <= 0) | F.col("nome_canal").isNull()
       | F.col("tipo_canal").isNull() | F.col("status_canal").isNull()
       | ~F.col("status_canal").isin("ativo", "inativo"), "Canal, tipo, status ou ID fora do contrato")
exigir(df_canais, "tb_canais", ((F.col("nivel_canal") == "Autosserviço") & (F.col("tipo_canal") != "Autosserviço"))
       | ((F.col("nivel_canal") != "Autosserviço") & (F.col("tipo_canal") != "Humano")), "Tipo do canal incompativel com o nivel")
validar_unicidade(df_canais, "tb_canais", ["nome_canal"])
gravar_silver(df_canais, "tb_canais")


## 2. Motivos

As três categorias ausentes seguem a classificação enviada para o case.
`chave_motivo` permite associar o texto livre, preservado depois em `motivo_original`.
Uma chave de associação ambígua bloqueia o join para não multiplicar chamados.

In [ ]:
categorias = {"financeiro": "Financeiro", "cartao": "Cartão", "cadastral": "Cadastral", "atendimento": "Atendimento", "beneficios": "Benefícios"}
categorias_sem_origem = {"contratacao de cartao adicional": "Cartão", "transferencia de agencia": "Cadastral", "cancelamento de conta": "Financeiro"}
criticidades = {"baixa": "Baixa", "media": "Média", "alta": "Alta"}

df = carregar("tb_motivos").withColumn("chave_motivo", chave_texto(F.col("nome_motivo")))
registrar_condicoes(df, "tb_motivos", [
    ("Categoria ausente classificada pela regra do case", "correcao", vazio(F.col("categoria"))),
    ("Grafia da categoria padronizada", "correcao", ~vazio(F.col("categoria")) & ~F.col("categoria").isin(*categorias.values())),
    ("Grafia da criticidade padronizada", "correcao", ~F.col("criticidade").isin(*criticidades.values())),
])
df_motivos = df.select(
    F.expr("try_cast(trim(id_motivo) AS INT)").alias("id_motivo"), F.trim("nome_motivo").alias("nome_motivo"),
    F.when(vazio(F.col("categoria")), de_para(categorias_sem_origem)[F.col("chave_motivo")])
    .otherwise(de_para(categorias)[chave_texto(F.col("categoria"))]).alias("categoria"),
    de_para(criticidades)[chave_texto(F.col("criticidade"))].alias("criticidade"), "chave_motivo", *metadados,
)
validar_unicidade(df_motivos, "tb_motivos", ["id_motivo"])
validar_unicidade(df_motivos, "tb_motivos", ["chave_motivo"])
exigir(df_motivos, "tb_motivos", (F.col("id_motivo") <= 0) | vazio(F.col("chave_motivo"))
       | F.col("categoria").isNull() | F.col("criticidade").isNull(), "Motivo ou classificacao fora do contrato")
gravar_silver(df_motivos, "tb_motivos")


## 3. Atendentes

O nível usa um de-para fechado: um valor como `12` não pode virar nível 1.
Datas inválidas não são confundidas com desligamento vazio. O status segue o
contrato enviado: com desligamento, inativo; sem desligamento, ativo.
Isso não exclui atendimentos históricos de pessoas hoje desligadas.

In [ ]:
niveis_atendentes = {"1": 1, "n1": 1, "nivel 1": 1, "2": 2, "n2": 2, "nivel 2": 2}
df = carregar("tb_atendentes")
registrar_condicoes(df, "tb_atendentes", [
    ("Nivel do atendente padronizado", "correcao", ~F.trim("nivel_atendimento").isin("1", "2")),
])
df_atendentes = df.select(
    F.expr("try_cast(trim(id_atendente) AS INT)").alias("id_atendente"), F.trim("nome_atendente").alias("nome_atendente"),
    de_para(niveis_atendentes)[chave_texto(F.col("nivel_atendimento"))].cast("int").alias("nivel_atendimento"),
    data_segura(F.col("data_admissao")).alias("data_admissao"),
    data_segura(F.col("data_desligamento")).alias("data_desligamento"),
    (~vazio(F.col("data_desligamento"))).alias("_desligamento_informado"),
    chave_texto(F.col("status_atendente")).alias("_status_origem"), *metadados,
).withColumn("status_atendente", F.when(F.col("data_desligamento").isNotNull(), "inativo").otherwise("ativo"))
registrar_condicoes(df_atendentes, "tb_atendentes", [
    ("Status recalculado divergente da origem", "correcao", ~F.col("_status_origem").eqNullSafe(F.col("status_atendente"))),
])
validar_unicidade(df_atendentes, "tb_atendentes", ["id_atendente"])
exigir(df_atendentes, "tb_atendentes", (F.col("id_atendente") <= 0) | vazio(F.col("nome_atendente"))
       | F.col("nivel_atendimento").isNull() | F.col("data_admissao").isNull()
       | (F.col("_desligamento_informado") & F.col("data_desligamento").isNull())
       | (F.col("data_desligamento") < F.col("data_admissao")), "Nivel ou datas do atendente invalidos")
df_atendentes = df_atendentes.drop("_desligamento_informado", "_status_origem")
gravar_silver(df_atendentes, "tb_atendentes")


## 4. Clientes

Preservamos a chave como texto de 11 dígitos. Não completamos uma chave curta
por suposição. Idade fora de 18–100 fica NULL; e-mail recebe a checagem básica
de presença de `@`, conforme o case. Isso não garante um endereço real ou válido.

In [ ]:
regioes = {"norte": "Norte", "nordeste": "Nordeste", "centro-oeste": "Centro-Oeste", "sudeste": "Sudeste", "sul": "Sul"}
df = carregar("tb_clientes").withColumn("_idade_numero", F.expr("try_cast(trim(idade) AS INT)"))
registrar_condicoes(df, "tb_clientes", [
    ("Grafia ou espacos da regiao padronizados", "correcao", ~F.col("regiao").isin(*regioes.values())),
    ("Idade ausente na origem", "alerta", vazio(F.col("idade"))),
    ("Idade informada invalida convertida para NULL", "correcao", ~vazio(F.col("idade")) & (F.col("_idade_numero").isNull() | ~F.col("_idade_numero").between(18, 100))),
    ("Email informado sem arroba convertido para NULL", "correcao", ~vazio(F.col("email")) & ~F.col("email").contains("@")),
    ("Data de cadastro ISO convertida", "informacao", F.trim("data_cadastro").rlike(r"^[0-9]{4}-[0-9]{2}-[0-9]{2}$")),
])
df_clientes = df.select(
    F.trim("id_cliente").alias("id_cliente"), F.trim("nome").alias("nome"),
    F.when(F.col("email").contains("@"), F.trim("email")).alias("email"),
    de_para(regioes)[chave_texto(F.col("regiao"))].alias("regiao"),
    F.when(F.col("_idade_numero").between(18, 100), F.col("_idade_numero")).alias("idade"),
    data_segura(F.col("data_cadastro")).alias("data_cadastro"),
    F.expr("try_cast(trim(dia_vencimento_fatura) AS INT)").alias("dia_vencimento_fatura"), *metadados,
).withColumn("faixa_etaria",
    F.when(F.col("idade") < 30, "18 a 29").when(F.col("idade") < 45, "30 a 44")
    .when(F.col("idade") < 60, "45 a 59").when(F.col("idade") < 75, "60 a 74")
    .when(F.col("idade").isNotNull(), "75 ou mais").otherwise("Não informado")
)
registrar_condicoes(df_clientes, "tb_clientes", [
    ("Vencimento invalido convertido para NULL", "correcao", F.col("dia_vencimento_fatura").isNull() | ~F.col("dia_vencimento_fatura").between(1, 31)),
])
df_clientes = df_clientes.withColumn("dia_vencimento_fatura", F.when(F.col("dia_vencimento_fatura").between(1, 31), F.col("dia_vencimento_fatura")))
validar_unicidade(df_clientes, "tb_clientes", ["id_cliente"])
exigir(df_clientes, "tb_clientes", ~F.col("id_cliente").rlike(r"^[0-9]{11}$")
       | F.col("regiao").isNull() | F.col("data_cadastro").isNull(), "Chave do cliente, regiao ou data invalidas")
gravar_silver(df_clientes, "tb_clientes")


## 5. Horários: fonte oficial do cliente e dos tempos

Abertura com ano posterior à finalização recebe o ano da finalização **somente
se a data resultante for válida e não ficar depois do início/fim**. Mantemos
as strings originais e uma flag da correção. A finalização que volta no relógio,
no mesmo dia do início, recebe um dia para representar a virada da meia-noite.

Desistência significa início vazio na origem. Erro de parsing é outra situação.
Tempos são segundos e só são calculados quando todos os horários estão presentes
e ordenados. Nas desistências, os dois tempos ficam NULL, conforme o case.

In [ ]:
df = carregar("tb_chamados_hora")
df_horas = df.select(
    F.expr("try_cast(trim(id_chamado) AS INT)").alias("id_chamado"),
    F.trim("id_cliente").alias("id_cliente"),
    F.col("hora_abertura_chamado").alias("hora_abertura_original"),
    F.col("hora_inicio_atendimento").alias("hora_inicio_original"),
    F.col("hora_finalizacao_atendimento").alias("hora_finalizacao_original"), *metadados,
)
for origem, destino in [
    ("hora_abertura_original", "hora_abertura_chamado"),
    ("hora_inicio_original", "hora_inicio_atendimento"),
    ("hora_finalizacao_original", "hora_finalizacao_atendimento"),
]:
    df_horas = df_horas.withColumn(destino, F.try_to_timestamp(F.trim(F.col(origem)), F.lit("dd/MM/yyyy 'às' HH:mm:ss")))

df_horas = (
    df_horas.withColumn("desistencia", vazio(F.col("hora_inicio_original")))
    .withColumn("_ano_posterior", F.coalesce(F.year("hora_abertura_chamado") > F.year("hora_finalizacao_atendimento"), F.lit(False)))
    .withColumn("_abertura_candidata", F.try_to_timestamp(
        F.concat(F.year("hora_finalizacao_atendimento").cast("string"), F.date_format("hora_abertura_chamado", "-MM-dd HH:mm:ss")),
        F.lit("yyyy-MM-dd HH:mm:ss")))
    .withColumn("flag_ano_abertura_corrigido", F.coalesce(
        F.col("_ano_posterior") & F.col("_abertura_candidata").isNotNull()
        & (F.col("_abertura_candidata") <= F.coalesce(F.col("hora_inicio_atendimento"), F.col("hora_finalizacao_atendimento"))), F.lit(False)))
    .withColumn("hora_abertura_chamado", F.when(F.col("flag_ano_abertura_corrigido"), F.col("_abertura_candidata")).otherwise(F.col("hora_abertura_chamado")))
    .withColumn("flag_virada_meia_noite", F.coalesce(
        (F.col("hora_finalizacao_atendimento") < F.col("hora_inicio_atendimento"))
        & (F.to_date("hora_finalizacao_atendimento") == F.to_date("hora_inicio_atendimento")), F.lit(False)))
    .withColumn("hora_finalizacao_atendimento", F.when(F.col("flag_virada_meia_noite"),
        F.expr("hora_finalizacao_atendimento + INTERVAL 1 DAY")).otherwise(F.col("hora_finalizacao_atendimento")))
    .withColumn("flag_horario_invalido", F.col("hora_abertura_chamado").isNull()
        | F.col("hora_finalizacao_atendimento").isNull()
        | (~F.col("desistencia") & F.col("hora_inicio_atendimento").isNull()))
    .withColumn("flag_cronologia_invalida", F.coalesce(
        (F.col("hora_inicio_atendimento") < F.col("hora_abertura_chamado"))
        | (F.col("hora_finalizacao_atendimento") < F.coalesce(F.col("hora_inicio_atendimento"), F.col("hora_abertura_chamado"))), F.lit(False)))
    .withColumn("tempos_validos", ~F.col("desistencia") & ~F.col("flag_horario_invalido") & ~F.col("flag_cronologia_invalida"))
    .withColumn("tempo_espera", F.when(F.col("tempos_validos"),
        F.col("hora_inicio_atendimento").cast("long") - F.col("hora_abertura_chamado").cast("long")))
    .withColumn("tempo_atendimento", F.when(F.col("tempos_validos"),
        F.col("hora_finalizacao_atendimento").cast("long") - F.col("hora_inicio_atendimento").cast("long")))
)
registrar_condicoes(df_horas, "tb_chamados_hora", [
    ("Ano da abertura corrigido com apoio da finalizacao", "correcao", F.col("flag_ano_abertura_corrigido")),
    ("Ano inconsistente sem correcao segura", "alerta", F.col("_ano_posterior") & ~F.col("flag_ano_abertura_corrigido")),
    ("Finalizacao ajustada na virada da meia-noite", "correcao", F.col("flag_virada_meia_noite")),
    ("Desistencia por inicio vazio na origem", "informacao", F.col("desistencia")),
    ("Horario obrigatorio ausente ou invalido", "alerta", F.col("flag_horario_invalido")),
    ("Cronologia inconsistente mantida sem calcular tempos", "alerta", F.col("flag_cronologia_invalida")),
    ("Tempos recalculados a partir dos horarios", "informacao", F.col("tempos_validos")),
])
df_horas = df_horas.drop("_ano_posterior", "_abertura_candidata")
validar_unicidade(df_horas, "tb_chamados_hora", ["id_chamado"])
exigir(df_horas, "tb_chamados_hora", (F.col("id_chamado") <= 0) | F.col("id_cliente").isNull()
       | ~F.col("id_cliente").rlike(r"^[0-9]{11}$"), "Chave do chamado ou do cliente invalida")
clientes_ausentes = df_horas.join(df_clientes.select("id_cliente"), "id_cliente", "left_anti")
exigir(clientes_ausentes, "tb_chamados_hora", F.lit(True), "Cliente dos horarios sem cadastro de referencia", df_horas.count())
gravar_silver(df_horas, "tb_chamados_hora")


## 6. Chamados: integração sem multiplicar linhas

Mantemos uma linha por chamado. Cliente, horários e tempos vêm de `tb_chamados_hora`;
os campos originais de horário/tempo de `tb_chamados` não participam do cálculo.
Motivo e canal são associados às referências por chaves únicas. Guardamos os
textos originais para explicar os de-paras.

Atendentes de autosserviço ficam NULL. ID inexistente em canal humano também fica
NULL e é sinalizado. Vínculo é comparado com a **data do atendimento**, inclusive
no dia de admissão/desligamento; status atual não exclui registros históricos.

In [ ]:
resolucoes = {"sim": "Sim", "s": "Sim", "1": "Sim", "nao": "Não", "n": "Não", "0": "Não", "nao resolvido": "Não"}
df = carregar("tb_chamados")
total_chamados = df.count()
df_chamados = df.select(
    F.expr("try_cast(trim(id_chamado) AS INT)").alias("id_chamado"),
    F.trim("id_cliente").alias("id_cliente_original"),
    F.col("motivo").alias("motivo_original"), F.col("canal").alias("canal_original"),
    F.col("resolvido").alias("resolvido_original"), F.col("id_atendente").alias("id_atendente_original"),
    F.expr("try_cast(trim(id_atendente) AS INT)").alias("_id_atendente_lido"), *metadados,
)
validar_unicidade(df_chamados, "tb_chamados", ["id_chamado"])

horas_integracao = df_horas.select(
    "id_chamado", "id_cliente", "hora_abertura_chamado", "hora_inicio_atendimento", "hora_finalizacao_atendimento",
    "tempo_espera", "tempo_atendimento", "desistencia", "tempos_validos", "flag_horario_invalido",
    "flag_cronologia_invalida", "flag_ano_abertura_corrigido", "flag_virada_meia_noite",
    F.col("arquivo_origem").alias("arquivo_origem_horarios"),
    F.col("ingestion_datetime").alias("ingestion_datetime_horarios"), F.lit(True).alias("_hora_localizada"),
)
df_chamados = df_chamados.join(horas_integracao, "id_chamado", "left")
exigir(df_chamados, "tb_chamados", F.col("_hora_localizada").isNull(), "Chamado sem registro na base oficial de horarios")

df_chamados = (
    df_chamados.withColumn("chave_motivo", chave_texto(F.col("motivo_original")))
    .withColumn("_nome_canal", de_para(nomes_canais)[F.regexp_replace(chave_texto(F.col("canal_original")), "[^a-z0-9]", "")])
    .withColumn("resolvido", F.coalesce(de_para(resolucoes)[chave_texto(F.col("resolvido_original"))], F.lit("Não informado")))
    .join(df_motivos.select("chave_motivo", "id_motivo", F.col("nome_motivo").alias("motivo")), "chave_motivo", "left")
    .join(df_canais.select(F.col("nome_canal").alias("_nome_canal"), "id_canal", "tipo_canal", "nivel_canal"), "_nome_canal", "left")
    .withColumn("canal", F.col("_nome_canal"))
    .join(df_atendentes.select(
        F.col("id_atendente").alias("_id_atendente_lido"), F.col("nivel_atendimento").alias("_nivel_atendente"),
        "data_admissao", "data_desligamento", F.lit(True).alias("_atendente_localizado")), "_id_atendente_lido", "left")
    .withColumn("flag_cliente_corrigido", ~F.col("id_cliente_original").eqNullSafe(F.col("id_cliente")))
    .withColumn("flag_atendente_inexistente", F.coalesce((F.col("tipo_canal") == "Humano") & F.col("_atendente_localizado").isNull(), F.lit(False)))
    .withColumn("flag_atendente_removido_autosservico", F.coalesce((F.col("tipo_canal") == "Autosserviço") & ~vazio(F.col("id_atendente_original")), F.lit(False)))
    .withColumn("id_atendente", F.when((F.col("tipo_canal") == "Humano") & F.col("_atendente_localizado").isNotNull(), F.col("_id_atendente_lido")))
    .withColumn("flag_vinculo_atendente_invalido", F.when(F.col("id_atendente").isNotNull() & F.col("hora_abertura_chamado").isNotNull(),
        F.coalesce((F.to_date("hora_abertura_chamado") < F.col("data_admissao"))
            | (F.to_date("hora_abertura_chamado") > F.col("data_desligamento")), F.lit(False))))
    .withColumn("flag_nivel_atendente_incompativel", F.when(F.col("id_atendente").isNotNull(),
        ((F.col("nivel_canal") == "N1") & (F.col("_nivel_atendente") != 1))
        | ((F.col("nivel_canal") == "N2") & (F.col("_nivel_atendente") != 2))))
    .withColumn("flag_resolvido_em_desistencia", F.col("desistencia") & (F.col("resolvido") == "Sim"))
    .withColumn("mes_referencia", F.trunc(F.to_date("hora_abertura_chamado"), "month"))
)
exigir(df_chamados, "tb_chamados", (F.col("id_chamado") <= 0) | F.col("id_canal").isNull()
       | F.col("id_motivo").isNull(), "ID invalido ou motivo/canal sem associacao de referencia")
registrar_condicoes(df_chamados, "tb_chamados", [
    ("Cliente substituido pela base oficial de horarios", "correcao", F.col("flag_cliente_corrigido")),
    ("Canal padronizado pelo de-para", "correcao", ~F.col("canal_original").eqNullSafe(F.col("canal"))),
    ("Grafia do motivo padronizada pela referencia", "correcao", ~F.col("motivo_original").eqNullSafe(F.col("motivo"))),
    ("Resolucao informada fora do dominio convertida para Nao informado", "alerta", ~vazio(F.col("resolvido_original")) & (F.col("resolvido") == "Não informado")),
    ("Resolucao ausente na origem", "alerta", vazio(F.col("resolvido_original"))),
    ("Grafia ou codigo da resolucao padronizado", "correcao", ~vazio(F.col("resolvido_original")) & ~F.col("resolvido_original").eqNullSafe(F.col("resolvido"))),
    ("Atendente removido de autosservico", "correcao", F.col("flag_atendente_removido_autosservico")),
    ("Atendente humano inexistente convertido para NULL", "alerta", F.col("flag_atendente_inexistente")),
    ("Atendimento fora do vinculo do atendente mantido", "alerta", F.col("flag_vinculo_atendente_invalido")),
    ("Nivel do atendente incompativel com o canal", "alerta", F.col("flag_nivel_atendente_incompativel")),
    ("Chamado resolvido com desistencia mantido para analise", "alerta", F.col("flag_resolvido_em_desistencia")),
], "apos_integracao")
df_chamados = df_chamados.select(
    "id_chamado", "id_cliente", "id_motivo", "id_canal", "id_atendente", "motivo", "canal", "tipo_canal", "nivel_canal", "resolvido",
    "hora_abertura_chamado", "hora_inicio_atendimento", "hora_finalizacao_atendimento", "tempo_espera", "tempo_atendimento",
    "desistencia", "tempos_validos", "mes_referencia", "motivo_original", "canal_original", "resolvido_original",
    "id_cliente_original", "id_atendente_original", "flag_cliente_corrigido", "flag_atendente_inexistente",
    "flag_atendente_removido_autosservico", "flag_vinculo_atendente_invalido", "flag_nivel_atendente_incompativel",
    "flag_resolvido_em_desistencia", "flag_horario_invalido", "flag_cronologia_invalida", "flag_ano_abertura_corrigido", "flag_virada_meia_noite",
    *metadados, "arquivo_origem_horarios", "ingestion_datetime_horarios",
)
validar_unicidade(df_chamados, "tb_chamados", ["id_chamado"])
if df_chamados.count() != total_chamados:
    raise ValueError("O join alterou o numero de chamados. Gold bloqueada.")
gravar_silver(df_chamados, "tb_chamados")


## 7. Custos: um custo por chamado

Removemos somente as repetições idênticas. Dois custos diferentes para o mesmo
chamado bloqueiam a carga: somar ou escolher um deles mudaria o significado da base.

O valor usa decimal(18,2). Aplicamos valor absoluto aos negativos pela **premissa
explícita deste case de erro de sinal**, guardando o original e a flag. Isso não
é uma regra geral para dados financeiros. Zero representa custo não registrado,
fica NULL e não deve ser tratado como atendimento gratuito na Gold.

In [ ]:
df = carregar("tb_custos").select(
    F.expr("try_cast(trim(id_custo) AS INT)").alias("id_custo"),
    F.expr("try_cast(trim(id_chamado) AS INT)").alias("id_chamado"),
    F.col("custo").alias("custo_original"), *metadados,
)
validar_unicidade(df, "tb_custos", ["id_custo"])
exigir(df, "tb_custos", (F.col("id_custo") <= 0) | F.col("id_chamado").isNull()
       | (F.col("id_chamado") <= 0), "Chave do custo ou do chamado invalida")
total_custos = df.count()
df_custos = df.join(df_chamados.select("id_chamado"), "id_chamado", "left_semi")
registrar("tb_custos", "Custo de chamado inexistente removido", total_custos - df_custos.count(), total_custos, "descarte", "integridade")
df_custos = (
    df_custos.withColumn("_custo_texto", F.regexp_replace(F.lower(F.trim("custo_original")), r"r\$|reais|\s", ""))
    .withColumn("_custo_texto", F.when(F.col("_custo_texto").contains(","),
        F.regexp_replace(F.regexp_replace("_custo_texto", r"\.", ""), ",", ".")).otherwise(F.col("_custo_texto")))
    .withColumn("_custo_numero", F.when(F.col("_custo_texto").rlike(r"^[+-]?[0-9]+(\.[0-9]+)?$"),
        F.expr("try_cast(_custo_texto AS DECIMAL(18,2))")))
    .withColumn("flag_custo_negativo_corrigido", F.coalesce(F.col("_custo_numero") < 0, F.lit(False)))
    .withColumn("flag_custo_zero_nao_registrado", F.coalesce(F.col("_custo_numero") == 0, F.lit(False)))
    .withColumn("flag_custo_invalido", F.col("_custo_numero").isNull())
    .withColumn("custo", F.when(F.col("_custo_numero") != 0, F.abs("_custo_numero")).cast("decimal(18,2)"))
)
registrar_condicoes(df_custos, "tb_custos", [
    ("Custo negativo corrigido pela premissa de erro de sinal", "correcao", F.col("flag_custo_negativo_corrigido")),
    ("Custo zero nao registrado convertido para NULL", "correcao", F.col("flag_custo_zero_nao_registrado")),
    ("Custo ausente ou formato invalido mantido como NULL", "alerta", F.col("flag_custo_invalido")),
])
df_custos = df_custos.select("id_custo", "id_chamado", "custo", "custo_original", "flag_custo_negativo_corrigido", "flag_custo_zero_nao_registrado", "flag_custo_invalido", *metadados)
validar_unicidade(df_custos, "tb_custos", ["id_chamado"])
gravar_silver(df_custos, "tb_custos")


## 8. Pesquisa de satisfação

A nota aceita inteiros de 1 a 5. `NULL` textual e vazio são ausência; -1, 0, 6
ou outro conteúdo inválido ficam NULL e são sinalizados. Se houver mais de uma
pesquisa do mesmo chamado **com a mesma nota**, mantemos o maior `id_pesquisa`.
Se as notas divergirem, o Job para: ID maior não comprova resposta mais recente.

Pesquisa em desistência é mantida e sinalizada, conforme a regra enviada.
`satisfeito` é TRUE para 4/5, FALSE para 1/2/3 e NULL sem nota válida, evitando
que a ausência de nota entre no denominador do CSAT como insatisfação.

In [ ]:
df = carregar("tb_pesquisa_satisfacao").select(
    F.expr("try_cast(trim(id_pesquisa) AS INT)").alias("id_pesquisa"),
    F.expr("try_cast(trim(id_chamado) AS INT)").alias("id_chamado"),
    F.col("nota_atendimento").alias("nota_original"), *metadados,
)
validar_unicidade(df, "tb_pesquisa_satisfacao", ["id_pesquisa"])
exigir(df, "tb_pesquisa_satisfacao", (F.col("id_pesquisa") <= 0) | F.col("id_chamado").isNull()
       | (F.col("id_chamado") <= 0), "Chave da pesquisa ou do chamado invalida")
total_pesquisas = df.count()
df_pesquisa = df.join(df_chamados.select("id_chamado"), "id_chamado", "left_semi")
registrar("tb_pesquisa_satisfacao", "Pesquisa de chamado inexistente removida", total_pesquisas - df_pesquisa.count(), total_pesquisas, "descarte", "integridade")
df_pesquisa = (
    df_pesquisa.withColumn("_nota_numero", F.expr("try_cast(trim(nota_original) AS INT)"))
    .withColumn("flag_nota_ausente", vazio(F.col("nota_original")) | (chave_texto(F.col("nota_original")) == "null"))
    .withColumn("nota_atendimento", F.when(F.col("_nota_numero").between(1, 5), F.col("_nota_numero")))
    .withColumn("flag_nota_invalida", ~F.col("flag_nota_ausente") & F.col("nota_atendimento").isNull())
    .withColumn("_chave_nota", F.when(F.col("flag_nota_ausente"), "SEM_NOTA")
        .when(F.col("nota_atendimento").isNotNull(), F.col("nota_atendimento").cast("string"))
        .otherwise(F.concat(F.lit("INVALIDA:"), F.trim("nota_original"))))
)
conflitos = df_pesquisa.groupBy("id_chamado").agg(F.countDistinct("_chave_nota").alias("notas_distintas")).filter(F.col("notas_distintas") > 1)
linhas_conflitantes = df_pesquisa.join(conflitos.select("id_chamado"), "id_chamado", "inner")
exigir(linhas_conflitantes, "tb_pesquisa_satisfacao", F.lit(True), "Pesquisas do mesmo chamado com notas conflitantes", df_pesquisa.count())

total_antes_consolidacao = df_pesquisa.count()
pesquisas_escolhidas = df_pesquisa.groupBy("id_chamado").agg(F.max("id_pesquisa").alias("id_pesquisa"))
df_pesquisa = (
    df_pesquisa.join(pesquisas_escolhidas, ["id_chamado", "id_pesquisa"], "inner")
    .join(df_chamados.select("id_chamado", "desistencia"), "id_chamado", "left")
    .withColumn("flag_pesquisa_em_desistencia", F.col("desistencia"))
    .withColumn("satisfeito", F.when(F.col("nota_atendimento").isNotNull(), F.col("nota_atendimento") >= 4))
)
registrar("tb_pesquisa_satisfacao", "Pesquisas redundantes com mesma nota consolidadas pelo maior ID", total_antes_consolidacao - df_pesquisa.count(), total_antes_consolidacao, "descarte", "consolidacao")
registrar_condicoes(df_pesquisa, "tb_pesquisa_satisfacao", [
    ("Nota ausente mantida como NULL", "alerta", F.col("flag_nota_ausente")),
    ("Nota fora de 1 a 5 ou formato invalido convertida para NULL", "correcao", F.col("flag_nota_invalida")),
    ("Pesquisa em desistencia mantida e sinalizada", "alerta", F.col("flag_pesquisa_em_desistencia")),
], "apos_consolidacao")
df_pesquisa = df_pesquisa.select("id_pesquisa", "id_chamado", "nota_atendimento", "satisfeito", "nota_original", "flag_nota_ausente", "flag_nota_invalida", "flag_pesquisa_em_desistencia", *metadados)
validar_unicidade(df_pesquisa, "tb_pesquisa_satisfacao", ["id_chamado"])
gravar_silver(df_pesquisa, "tb_pesquisa_satisfacao")


## 9. Log e conferência final

O log usa a quantidade de linhas da etapa indicada como denominador. Regras
podem atingir a mesma linha: seus percentuais **não podem ser somados** para
calcular um índice de qualidade. Ausência legítima, correção e bloqueio são
situações diferentes. Overwrite guarda o diagnóstico da última execução.

Uma falha de conteúdo validada por `exigir` salva o diagnóstico antes de parar;
falhas de conexão, permissões ou escrita continuam registradas na saída do Job.

In [ ]:
salvar_log()
display(spark.table(f"{silver}.tb_log_qualidade").orderBy("base", "etapa", "regra"))

for tabela in ["tb_canais", "tb_motivos", "tb_atendentes", "tb_clientes", "tb_chamados_hora", "tb_chamados", "tb_custos", "tb_pesquisa_satisfacao"]:
    print(f"{silver}.{tabela}: {spark.table(f'{silver}.{tabela}').count()} linhas.")


## Contrato para a Gold

`tb_chamados` tem uma linha por chamado; custos e pesquisas têm no máximo uma
linha por chamado, mas podem não cobrir todos eles. Use LEFT JOIN para manter
os chamados sem custo ou nota. Não transforme esses NULLs em zero.
TME/TMA usam somente `tempos_validos`; CSAT usa somente notas válidas.
Jornadas, fatos, dimensões, taxas e exportações para Power BI ficam na Gold.